In [1]:
#pip install huggingface_hub

In [2]:
from huggingface_hub import list_repo_files, hf_hub_download
import pandas as pd
import numpy as np

In [3]:
REPO_ID = "Aneeshers/tennis-sackmann-archive"

In [4]:
path = hf_hub_download(
    repo_id=REPO_ID,
    filename="slam_pointbypoint/2019-usopen-points.csv",
    repo_type="dataset"
)

In [5]:
matches_path = hf_hub_download(
    repo_id="Aneeshers/tennis-sackmann-archive",
    filename="slam_pointbypoint/2019-usopen-matches.csv",
    repo_type="dataset"
)

In [6]:
matches_df = pd.read_csv(matches_path)
matches_df.head()

,match_id,year,slam,match_num,player1,player2,status,winner,event_name,round,court_name,court_id,player1id,player2id,nation1,nation2
0,2019-usopen-1101,2019,usopen,1101,Novak Djokovic,Roberto Carballes Baena,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-usopen-1102,2019,usopen,1102,Sam Querrey,Juan Ignacio Londero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-usopen-1103,2019,usopen,1103,Denis Kudla,Janko Tipsarevic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-usopen-1104,2019,usopen,1104,Steve Darcis,Dusan Lajovic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-usopen-1105,2019,usopen,1105,Stan Wawrinka,Jannik Sinner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df = pd.read_csv(path)

In [8]:
mask = pd.to_numeric(df['PointNumber'], errors='coerce').notna()

In [9]:
df = df[mask]

In [10]:
df = pd.merge(
    df,
    matches_df[['match_id','player1','player2']],
    on="match_id",
    how="inner"
)

In [11]:
df['PointNumber'] = pd.to_numeric(df['PointNumber'], errors='coerce')

In [12]:
idx = df.groupby('match_id')['PointNumber'].idxmax()
last_points = df.loc[idx]

In [13]:
retired_match_ids = last_points[last_points['SetWinner'] == 0]['match_id']

# Remove ALL rows belonging to those matches from the full df
df_clean = df[~df['match_id'].isin(retired_match_ids)].copy()

print(f"Removed {len(retired_match_ids)} matches, {len(df) - len(df_clean)} rows")

Removed 5 matches, 612 rows


In [14]:
winners = last_points[['match_id', 'PointWinner']]

In [15]:
winners = winners.rename(columns={'PointWinner': 'MatchWinner'})

In [16]:
df_clean = pd.merge(
    df_clean,
    winners,
    on='match_id',
    how='left'
)

In [17]:
df_clean.columns

Index(['match_id', 'ElapsedTime', 'SetNo', 'P1GamesWon', 'P2GamesWon',
       'SetWinner', 'GameNo', 'GameWinner', 'PointNumber', 'PointWinner',
       'PointServer', 'Speed_KMH', 'Rally', 'P1Score', 'P2Score', 'P1Momentum',
       'P2Momentum', 'P1PointsWon', 'P2PointsWon', 'P1Ace', 'P2Ace',
       'P1Winner', 'P2Winner', 'P1DoubleFault', 'P2DoubleFault', 'P1UnfErr',
       'P2UnfErr', 'P1NetPoint', 'P2NetPoint', 'P1NetPointWon',
       'P2NetPointWon', 'P1BreakPoint', 'P2BreakPoint', 'P1BreakPointWon',
       'P2BreakPointWon', 'P1FirstSrvIn', 'P2FirstSrvIn', 'P1FirstSrvWon',
       'P2FirstSrvWon', 'P1SecondSrvIn', 'P2SecondSrvIn', 'P1SecondSrvWon',
       'P2SecondSrvWon', 'P1ForcedError', 'P2ForcedError', 'History',
       'Speed_MPH', 'P1BreakPointMissed', 'P2BreakPointMissed',
       'ServeIndicator', 'Serve_Direction', 'Winner_FH', 'Winner_BH',
       'ServingTo', 'P1TurningPoint', 'P2TurningPoint', 'ServeNumber',
       'WinnerType', 'WinnerShotType', 'P1DistanceRun', 'P2Dista

In [18]:
set_results = df_clean[df_clean['SetWinner'] != 0][['match_id','SetNo','SetWinner']]

In [19]:
rows = []
for match_id, match_df in set_results.groupby('match_id'):
    P1SetsWon = 0
    P2SetsWon = 0
    SetNo = 1
    for idx, i in set_results[set_results['match_id'] == match_id].iterrows():
        SetWinner = pd.to_numeric(i['SetWinner'], errors='coerce')
        rows.append({'match_id': match_id, 'SetNo': SetNo, 'P1SetsWon': P1SetsWon, 'P2SetsWon': P2SetsWon})
        if SetWinner == 1:
            P1SetsWon += 1
        elif SetWinner == 2:
            P2SetsWon += 1
        SetNo += 1

In [20]:
set_winners = pd.DataFrame(rows)

In [21]:
df_clean = pd.merge(
    df_clean,
    set_winners,
    on = ['match_id','SetNo'],
    how = 'left'
)

In [22]:
REGULAR_SCORES = {'15', '30', '40', 'AD'}

df_clean['IsTiebreak'] = (
    ~df_clean['P1Score'].astype(str).isin(REGULAR_SCORES)
    & ~df_clean['P2Score'].astype(str).isin(REGULAR_SCORES)
    & (df_clean['P1GamesWon'] == 6)
    & (df_clean['P2GamesWon'] == 6)
)

In [23]:
df_clean['GameScoreState'] = np.where(
    ~df_clean['IsTiebreak'],
    df_clean['P1Score'].astype(str) + '-' + df_clean['P2Score'].astype(str),
    np.nan
)

game_dummies = pd.get_dummies(df_clean['GameScoreState'], prefix='game_score', drop_first=True)

In [24]:
df_clean['TB_PointDiff'] = np.where(
    df_clean['IsTiebreak'],
    pd.to_numeric(df_clean['P1Score'], errors='coerce') - pd.to_numeric(df_clean['P2Score'], errors='coerce'),
    0
)
df_clean['TB_PointsPlayed'] = np.where(
    df_clean['IsTiebreak'],
    pd.to_numeric(df_clean['P1Score'], errors='coerce') + pd.to_numeric(df_clean['P2Score'], errors='coerce'),
    0
)

In [25]:
df_clean['GameScoreState'] = np.where(
    ~df_clean['IsTiebreak'],
    df_clean['P1Score'].astype(str) + '-' + df_clean['P2Score'].astype(str),
    np.nan
)

game_dummies = pd.get_dummies(df_clean['GameScoreState'], prefix='game_score', drop_first=True)

In [26]:
df_clean = pd.concat([df_clean, game_dummies], axis=1)

In [27]:
idx = set_winners.groupby('match_id')['SetNo'].idxmax()
final_sets = set_winners.loc[idx]
final_sets['WinningSets'] = final_sets[['P1SetsWon', 'P2SetsWon']].max(axis=1)
final_sets['BestOf'] = np.where(final_sets['WinningSets'] == 2, 5,
                          np.where(final_sets['WinningSets'] == 1, 3, np.nan))

In [28]:
df_clean = pd.merge(
    df_clean,
    final_sets[['match_id','BestOf']],
    on='match_id',
    how='left'
)

In [38]:
df_clean = df_clean[
    ~df_clean['match_id'].isin(
        final_sets.loc[final_sets['BestOf'].isna(), 'match_id']
    )
]

In [41]:
score_cols = df_clean.columns[df_clean.columns.str.lower().str.startswith('game_score')]
base_cols = ['match_id','player1','player2','PointServer','P1SetsWon','P2SetsWon',
             'P1GamesWon','P2GamesWon','IsTiebreak','P1Score','P2Score','BestOf','MatchWinner']

all_cols = base_cols + [c for c in score_cols if c not in base_cols]
df_clean[all_cols]

,match_id,player1,player2,PointServer,P1SetsWon,P2SetsWon,P1GamesWon,P2GamesWon,IsTiebreak,P1Score,...,game_score_30-0,game_score_30-15,game_score_30-30,game_score_30-40,game_score_40-0,game_score_40-15,game_score_40-30,game_score_40-40,game_score_40-AD,game_score_AD-40
0,2019-usopen-1101,Novak Djokovic,Roberto Carballes Baena,1,0,0,0,0,False,15,...,False,False,False,False,False,False,False,False,False,False
1,2019-usopen-1101,Novak Djokovic,Roberto Carballes Baena,1,0,0,0,0,False,15,...,False,False,False,False,False,False,False,False,False,False
2,2019-usopen-1101,Novak Djokovic,Roberto Carballes Baena,1,0,0,0,0,False,30,...,False,True,False,False,False,False,False,False,False,False
3,2019-usopen-1101,Novak Djokovic,Roberto Carballes Baena,1,0,0,0,0,False,30,...,False,False,True,False,False,False,False,False,False,False
4,2019-usopen-1101,Novak Djokovic,Roberto Carballes Baena,1,0,0,0,0,False,40,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46774,2019-usopen-2701,Bianca Andreescu,Serena Williams,2,1,0,6,5,False,15,...,False,False,False,False,False,False,False,False,False,False
46775,2019-usopen-2701,Bianca Andreescu,Serena Williams,2,1,0,6,5,False,30,...,False,True,False,False,False,False,False,False,False,False
46776,2019-usopen-2701,Bianca Andreescu,Serena Williams,2,1,0,6,5,False,40,...,False,False,False,False,False,True,False,False,False,False
46777,2019-usopen-2701,Bianca Andreescu,Serena Williams,2,1,0,6,5,False,40,...,False,False,False,False,False,False,True,False,False,False


In [42]:
df_clean[all_cols].to_csv('df.csv')